# Garnet–Clinopyroxene Fe²⁺–Mg Exchange Thermometry (S10 Eclogite)

Garnet and clinopyroxene coexist over a wide range of high-pressure mafic rocks, from
granulites to eclogites, and the Fe²⁺–Mg distribution between them is a long-established
thermometer. The exchange reaction

$$
\tfrac{1}{3}\mathrm{Mg_3Al_2Si_3O_{12}}\ (\text{pyrope}) + \mathrm{CaFeSi_2O_6}\ (\text{hedenbergite})
\;=\;
\tfrac{1}{3}\mathrm{Fe_3Al_2Si_3O_{12}}\ (\text{almandine}) + \mathrm{CaMgSi_2O_6}\ (\text{diopside})
$$

has an equilibrium distribution coefficient

$$
K_D = \frac{(\mathrm{Fe^{2+}/Mg})^{\mathrm{garnet}}}{(\mathrm{Fe^{2+}/Mg})^{\mathrm{clinopyroxene}}}
$$

whose temperature dependence is the basis of the thermometer: $K_D$ falls as temperature rises.

This notebook:

1. sets up the S10 eclogite bulk composition in the `mpe` database;
2. runs a P–T grid and extracts the garnet and clinopyroxene compositions;
3. maps $K_D$ across the grid using the traditional mixed Fe convention: garnet Fe is taken from its FeO-basis APFU (all Fe treated as Fe²⁺), while cpx Fe²⁺ comes from the excess-O heuristic split;
4. converts $K_D$ to temperature with Mysen & Heier (1972), Ganguly (1979), Krogh Ravna (2000), Ellis & Green (1979), and Räheim & Green (1974), and compares them against the true grid temperature.

---

## 1. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from phasetools import MAGEMinPTGridCalculator

## 2. Bulk composition and database

The S10 eclogite bulk composition (in mol%) is taken from the `phasetools` test suite. The
`mpe` database combines the metapelite (`mp`) and metabasite (`mb`) solution models and is
the appropriate choice for a mafic bulk at eclogite-facies conditions. Pressures are in kbar
and temperatures in °C throughout.

In [ ]:
# Eclogite S10 bulk composition in mol% (from the phasetools test suite).
Xoxides = ["H2O", "SiO2", "Al2O3", "CaO", "MgO", "FeO", "K2O", "Na2O", "TiO2", "MnO", "O"]
X = [0.92, 54.57, 8.79, 11.20, 8.45, 12.89, 0.24, 2.24, 1.12, 0.22, 0.64]

db = "mpe"          # cross-lithology metapelite + metabasite solution models
dataset = 636
sys_in = "mol"

calc = MAGEMinPTGridCalculator(db=db, dataset=dataset)
calc.setup_bulk_composition(Xoxides, X, sys_in=sys_in)

## 3. P–T grid

A grid spanning 15–40 kbar and 550–850 °C covers the eclogite-facies field where garnet and
clinopyroxene are both stable for this bulk composition.

In [ ]:
P = np.linspace(5.0, 20.0, 11)    # kbar
T = np.linspace(350.0, 900.0, 13)  # °C
Pgrid, Tgrid = np.meshgrid(P, T, indexing="xy")

out = calc.calculate_grid(Pgrid.ravel(), Tgrid.ravel())
print(f"{len(out)} grid points; unique phases: {calc.get_all_unique_phases(out)}")

## 4. Resolve the clinopyroxene phase name

MAGEMin reports clinopyroxene under different solution-model names depending on composition
(`dio` diopside, `omph` omphacite, `aug` augite). The name is resolved dynamically rather than
hard-coded, so the notebook follows whichever clinopyroxene solution is stable.

In [ ]:
CPX_CANDIDATES = ["dio", "omph", "aug", "jac"]  # clinopyroxene solution names in mpe

unique_phases = calc.get_all_unique_phases(out)
cpx = next((p for p in CPX_CANDIDATES if p in unique_phases), None)
if cpx is None:
    raise RuntimeError("No clinopyroxene phase found in the grid.")
print("Clinopyroxene phase resolved to:", cpx)

## 5. Extract Fe²⁺/Mg and compute $K_D$

Only **divalent** iron takes part in the Fe²⁺–Mg exchange. For the traditional calibration convention
used here, garnet Fe is deliberately taken as $b_0[\text{ox\_apfu\_FeO}]$ (the FeO basis, treating
all garnet Fe as Fe²⁺), whereas cpx Fe²⁺ is taken from the excess-O heuristic split. This mixed
convention is an intentional calibration assumption. Grid points where a phase
is absent — or where a solvus yields two coexisting limbs of the same phase, making the pairing
with the other phase ambiguous — are masked out.

In [ ]:
def extract_fe2_mg(calc, phase, grid_out):
    """Return (Fe2+/Mg, X_Ca, X_Mn, mask) arrays over the grid for ``phase``.

    Garnet uses ``ox_apfu_FeO`` deliberately: this is the traditional FeO-basis
    convention, treating all garnet Fe as Fe2+. Cpx uses ``Fe2`` from the
    excess-O heuristic split. This mixed convention is an intentional calibration
    assumption, not a claim that garnet Fe is strictly all Fe2+. Grid points where
    the phase is absent, or where a solvus
    gives two coexisting limbs of the same phase (an ambiguous pairing with the
    other phase), are flagged True in ``mask`` and should be excluded before
    forming a distribution coefficient.

    Parameters
    ----------
    calc : MAGEMinPTGridCalculator
        Calculator that produced ``grid_out``.
    phase : str
        Phase key (e.g. ``"g"``).
    grid_out : list
        Output of :meth:`MAGEMinPTGridCalculator.calculate_grid`.

    Returns
    -------
    fe2_mg, x_ca, x_mn, mask : numpy.ndarray
        Arrays over the grid. ``x_ca`` and ``x_mn`` are the garnet-style cation
        fractions Ca/(Ca+Mn+Fe+Mg) and Mn/(Ca+Mn+Fe+Mg), with the Fe
        convention above.
    """
    bundles = calc.extract_from_grid(
        phase, oxides=["MgO", "FeO", "CaO", "MnO"], fe_split=True, grid_out=grid_out
    )
    n = len(grid_out)
    if not bundles:  # phase never appears in the grid
        nan = np.full(n, np.nan)
        return nan, nan, nan, np.ones(n, dtype=bool)

    b0 = bundles[0]
    mg = b0["ox_apfu_MgO"]
    # if phase == 'g':
    #     # Traditional garnet FeO-basis convention: all garnet Fe is Fe2+.
    #     fe2 = b0["ox_apfu_FeO"]
    # else:
    fe2 = b0["Fe2"]
    ca = b0["ox_apfu_CaO"]
    mn = b0["ox_apfu_MnO"]

    denom = ca + mn + fe2 + mg          # Ca + Mn + Fe2+ + Mg, in atoms per formula unit
    with np.errstate(divide="ignore", invalid="ignore"):
        fe2_mg = fe2 / mg
        x_ca = ca / denom
        x_mn = mn / denom

    absent = np.isnan(fe2) | np.isnan(mg) | (mg <= 0.0)
    if len(bundles) > 1:
        ambiguous = ~np.isnan(bundles[1]["Fe2"])  # a second limb exists -> ambiguous pairing
    else:
        ambiguous = np.zeros(n, dtype=bool)
    return fe2_mg, x_ca, x_mn, absent | ambiguous

In [ ]:
fe2_mg_g, x_ca_g, x_mn_g, mask_g = extract_fe2_mg(calc, "g", out)
fe2_mg_c, x_ca_c, x_mn_c, mask_c = extract_fe2_mg(calc, cpx, out)

valid = ~mask_g & ~mask_c

Kd = np.full_like(fe2_mg_g, np.nan)
Kd[valid] = fe2_mg_g[valid] / fe2_mg_c[valid]

print(f"K_D defined at {int(valid.sum())} / {valid.size} grid points")
print(f"K_D range: {np.nanmin(Kd):.2f} – {np.nanmax(Kd):.2f}")

## 6. Map $K_D$ over the P–T grid

$K_D$ decreases smoothly with rising temperature — the signature of the Fe²⁺–Mg exchange
equilibrium — and is only weakly dependent on pressure, which is why a pressure-independent
calibration such as Mysen & Heier (1972) can work at all.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5))
Kd_map = Kd.reshape(Tgrid.shape)

cs = ax.contourf(Tgrid, Pgrid, Kd_map, levels=14, cmap="viridis")
ax.contour(Tgrid, Pgrid, Kd_map, levels=14, colors="k", linewidths=0.4)
cbar = fig.colorbar(cs, ax=ax, label=r"$K_D = (Fe^{2+}/Mg)^g \,/\, (Fe^{2+}/Mg)^{cpx}$")

ax.set_xlabel("Temperature (°C)")
ax.set_ylabel("Pressure (kbar)")
ax.set_title("Fe²⁺–Mg distribution coefficient, S10 eclogite (mpe)")
ax.grid(ls=":", alpha=0.4)
plt.show()

## 7. Thermometer calibrations

Five calibrations of the Fe²⁺–Mg exchange are implemented as pure-NumPy helpers. All return °C.

**Mysen & Heier (1972)** — a pressure- and composition-independent calibration
(following Banno, 1970):

$$
T\,(\mathrm{K}) = \frac{2475}{\ln K_D + 0.781}
$$

**Ganguly (1979)** — includes pressure and garnet grossular content. This notebook retains the
piecewise coefficients tabulated by Yavuz & Yıldırım (2020), rather than silently attributing
that printed piecewise form to the original Ganguly equation.
in the piecewise form tabulated by Yavuz & Yıldırım (2020):

$$
T\,(°\mathrm{C}) = \frac{4100 + 1586\,X_\mathrm{Ca}^{g} + 11.07\,P}{\ln K_D + 2.40} - 273.15
\quad (T \geq 1060\,°\mathrm{C})
$$

$$
T\,(°\mathrm{C}) = \frac{4801 + 1586\,X_\mathrm{Ca}^{g} + 11.07\,P}{\ln K_D + 2.93} - 273.15
\quad (T \leq 1060\,°\mathrm{C})
$$

where $X_\mathrm{Ca}^{g} = \mathrm{Ca}/(\mathrm{Ca} + \mathrm{Mn} + \mathrm{Fe^{2+}} + \mathrm{Mg})$
in garnet and $P$ is in kbar.

**Krogh Ravna (2000)** uses $P_{GPa}=P_{kbar}/10$, $X_{Ca}=Ca/(Ca+Mn+Fe+Mg)$,
$X_{Mn}=Mn/(Ca+Mn+Fe+Mg)$, and $X_{Mg\#}=Mg/(Mg+Fe)$.

**Ellis & Green (1979)** uses $X_{Ca,binary}=Ca/(Ca+Fe+Mg)$, deliberately excluding Mn;
this differs from the garnet site $X_{Ca}$ used by Krogh Ravna and Ganguly.

**Räheim & Green (1974)** is pressure-dependent but composition-independent.

In [ ]:
def T_mysen_heier_1972(Kd):
    """Mysen & Heier (1972) garnet–cpx Fe2+–Mg thermometer, in °C.

    T(K) = 2475 / (ln Kd + 0.781)
    """
    lnKd = np.log(Kd)
    return 2475.0 / (lnKd + 0.781) - 273.15


def T_ganguly_1979(Kd, P_kbar, X_Ca_grt):
    """Ganguly (1979) garnet–cpx Fe2+–Mg thermometer, in °C.

    Piecewise form tabulated by Yavuz & Yıldırım (2020, WinGrt):
      T >= 1060 °C:  T = (4100 + 1586·X_Ca + 11.07·P) / (ln Kd + 2.40) - 273.15
      T <= 1060 °C:  T = (4801 + 1586·X_Ca + 11.07·P) / (ln Kd + 2.93) - 273.15
    """
    lnKd = np.log(Kd)
    P = np.asarray(P_kbar, dtype=float)
    XCa = np.asarray(X_Ca_grt, dtype=float)
    T_hi = (4100.0 + 1586.0 * XCa + 11.07 * P) / (lnKd + 2.40) - 273.15
    T_lo = (4801.0 + 1586.0 * XCa + 11.07 * P) / (lnKd + 2.93) - 273.15
    return np.where(T_hi >= 1060.0, T_hi, T_lo)


def T_krogh_ravna_2000(Kd, P_kbar, X_Ca, X_Mn, X_Mg_number):
    """Krogh Ravna (2000) garnet-cpx thermometer, in °C."""
    lnKd = np.log(Kd)
    P_GPa = np.asarray(P_kbar, dtype=float) / 10.0
    numerator = (1939.9 + 3270.0 * X_Ca - 1396.0 * X_Ca**2
                 + 3319.0 * X_Mn + 3535.0 * X_Mn**2
                 + 1105.0 * X_Mg_number - 3561.0 * X_Mg_number**2
                 + 2324.0 * X_Mg_number**3 + 169.4 * P_GPa)
    return numerator / (lnKd + 1.223) - 273.15


def T_ellis_green_1979(Kd, P_kbar, X_Ca_binary):
    """Ellis & Green (1979) garnet-cpx thermometer, in °C."""
    return (3104.0 * X_Ca_binary + 3030.0 + 10.86 * P_kbar) / (np.log(Kd) + 1.9034) - 273.15


def T_raheim_green_1974(Kd, P_kbar):
    """Räheim & Green (1974) garnet-cpx thermometer, in °C."""
    return (3686.0 + 28.35 * P_kbar) / (np.log(Kd) + 2.33) - 273.15

## 8. Apply the thermometers and compare with the true temperature

Because every grid point has a known temperature, we can feed the modelled $K_D$ back into each
thermometer and see how well it recovers the input. Points lie above the 1:1 line when a
calibration overestimates the temperature.

In [ ]:
P_flat = Pgrid.ravel()
T_true = Tgrid.ravel()

T_MH = T_mysen_heier_1972(Kd)
T_G = T_ganguly_1979(Kd, P_flat, x_ca_g)
X_mg_number_g = fe2_mg_g / (1.0 + fe2_mg_g)
X_ca_binary_g = x_ca_g / (1.0 - x_mn_g)
T_KR = T_krogh_ravna_2000(Kd, P_flat, x_ca_g, x_mn_g, X_mg_number_g)
T_EG = T_ellis_green_1979(Kd, P_flat, X_ca_binary_g)
T_RG = T_raheim_green_1974(Kd, P_flat)
thermometers = {
    "Mysen & Heier (1972)": T_MH,
    "Ganguly (1979; Yavuz & Yıldırım piecewise)": T_G,
    "Krogh Ravna (2000)": T_KR,
    "Ellis & Green (1979)": T_EG,
    "Räheim & Green (1974)": T_RG,
}

fig, ax = plt.subplots(figsize=(7, 5.5))
ax.plot([Tgrid.min(), Tgrid.max()], [Tgrid.min(), Tgrid.max()], "k--", lw=1, label="1:1 (true T)")
markers = ["o", "^", "s", "D", "v"]
for (name, estimate), marker in zip(thermometers.items(), markers):
    ax.scatter(T_true[valid], estimate[valid], s=16, alpha=0.65, marker=marker, label=name)
ax.set_xlabel("True grid temperature (°C)")
ax.set_ylabel("Thermometer temperature (°C)")
ax.legend()
ax.grid(ls=":", alpha=0.4)
plt.show()

print("\nThermometer offsets on valid masked points:")
print(f"{'Calibration':42s} {'Mean':>8s} {'Median':>8s} {'RMSE':>8s}")
for name, estimate in thermometers.items():
    offset = estimate[valid] - T_true[valid]
    print(f"{name:42s} {np.mean(offset):+8.1f} {np.median(offset):+8.1f} {np.sqrt(np.mean(offset**2)):8.1f}")

fig, ax = plt.subplots(figsize=(8, 4))
names = list(thermometers)
offsets = [np.mean(thermometers[name][valid] - T_true[valid]) for name in names]
ax.barh(names, offsets, color=plt.get_cmap("viridis")(np.linspace(0.15, 0.85, len(names))))
ax.axvline(0.0, color="k", lw=0.8)
ax.set_xlabel("Mean thermometer offset (°C)")
ax.set_title("Calibration offsets on valid grid points")
ax.grid(axis="x", ls=":", alpha=0.4)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5))
Kd_map = Kd.reshape(Tgrid.shape)
T_RG_grid = T_RG.reshape(Tgrid.shape)

cs = ax.contourf(Tgrid, Pgrid, T_RG_grid - Tgrid, levels=14, cmap="viridis")
# ax.contour(Tgrid, Pgrid, Kd_map, levels=14, colors="k", linewidths=0.4)
cbar = fig.colorbar(cs, ax=ax, label=r"$\Delta$T (T$_{RG}$ $-$ T$_{MAGEMin}$)")

ax.set_xlabel("Temperature (°C)")
ax.set_ylabel("Pressure (kbar)")
ax.set_title("Fe²⁺–Mg distribution coefficient, S10 eclogite (mpe)")
ax.grid(ls=":", alpha=0.4)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5))
Kd_map = Kd.reshape(Tgrid.shape)
T_EG_grid = T_EG.reshape(Tgrid.shape)

cs = ax.contourf(Tgrid, Pgrid, T_EG_grid - Tgrid, levels=14, cmap="viridis")
# ax.contour(Tgrid, Pgrid, Kd_map, levels=14, colors="k", linewidths=0.4)
cbar = fig.colorbar(cs, ax=ax, label=r"$\Delta$T (T$_{RG}$ $-$ T$_{MAGEMin}$)")

ax.set_xlabel("Temperature (°C)")
ax.set_ylabel("Pressure (kbar)")
ax.set_title("Fe²⁺–Mg distribution coefficient, S10 eclogite (mpe)")
ax.grid(ls=":", alpha=0.4)
plt.show()

## 9. Why divalent iron — and why the calibrations disagree

Clinopyroxene can carry substantial Fe³⁺ (in acmite/jadeite-type components), so its Fe²⁺
value is taken from the excess-O heuristic split. Garnet is intentionally different here: the
traditional thermometer convention uses $b_0[\text{ox\_apfu\_FeO}]$, i.e. garnet Fe on an FeO
basis with all garnet Fe treated as Fe²⁺. The resulting garnet-FeO / cpx-heuristic-Fe²⁺ pairing
is a mixed convention adopted for calibration comparison, not a claim that both phases have
the same redox treatment.

In [ ]:
from phasetools import get_phase_fe_split

i0 = int(np.flatnonzero(valid)[0])  # first valid grid point
o0 = out[i0]
cpx_split = get_phase_fe_split(o0, cpx)
grt_feo_basis = calc.extract_from_grid("g", oxides=["FeO"], grid_out=[o0])[0]["ox_apfu_FeO"][0]

print(f"At {T_true[i0]:.0f} °C, {P_flat[i0]:.0f} kbar:")
print(f"  garnet     : FeO-basis Fe = {grt_feo_basis:.3f} (all treated as Fe2+ by convention)")
print(f"  {cpx:10s}: Fe2+ = {cpx_split['Fe2']:.3f}, Fe3+ = {cpx_split['Fe3']:.3f} "
      f"(Fe3+/FeOt = {cpx_split['Fe3']/(cpx_split['Fe2']+cpx_split['Fe3']):.0%})")

The calibrations disagree because they encode different pressure and composition corrections. Mysen & Heier
(1972) is pressure- and composition-independent; Ganguly (1979) is retained here only with its
Yavuz & Yıldırım (2020) piecewise coefficients explicitly labelled; Krogh Ravna (2000) includes
Mn, Ca, Mg#, and pressure; Ellis & Green (1979) uses binary garnet Ca and pressure; and Räheim &
Green (1974) uses pressure alone. The offsets therefore illustrate calibration uncertainty, compounded
by the intentional mixed garnet FeO / cpx heuristic Fe²⁺ convention. These equations should not be
extrapolated beyond their experimental compositional and P–T ranges.

---
## References

- Mysen, B. O., & Heier, K. S. (1972). Petrogenesis of eclogites in high grade metamorphic
  gneisses, exemplified by the Hareidland eclogite, western Norway. *Contributions to Mineralogy
  and Petrology*, 36(1), 73–94. https://doi.org/10.1007/BF00372836
- Ganguly, J. (1979). Garnet and clinopyroxene solid solutions, and geothermometry based on
  Fe–Mg distribution coefficient. *Geochimica et Cosmochimica Acta*, 43(7), 1021–1029.
  https://doi.org/10.1016/0016-7037(79)90091-7
- Yavuz, F., & Yıldırım, D. K. (2020). WinGrt, a Windows program for garnet supergroup minerals.
  *Journal of Geosciences*, 65(2), 71–95. https://doi.org/10.3190/jgeosci.303 (source of the
  Ganguly 1979 piecewise coefficients)
- Johnson, C. A., Bohlen, S. R., & Essene, E. J. (1983). An evaluation of garnet-clinopyroxene
  geothermometry in granulites. *Contributions to Mineralogy and Petrology*, 84(2–3), 191–198.
  https://doi.org/10.1007/BF00371285
- Krogh Ravna, E. (2000). The garnet–clinopyroxene Fe²⁺–Mg geothermometer: an updated
  calibration. *Journal of Metamorphic Geology*, 18(2), 211–219.
  https://doi.org/10.1046/j.1525-1314.2000.00247.x
- Ellis, D. J., & Green, D. H. (1979). An experimental study of the effect of Ca upon garnet–
  clinopyroxene Fe–Mg exchange equilibria. *Contributions to Mineralogy and Petrology*, 71, 13–22.
  https://doi.org/10.1007/BF00371878
- Räheim, A., & Green, D. H. (1974). Experimental determination of the temperature and pressure
  dependence of the Fe–Mg partition coefficient for coexisting garnet and clinopyroxene.
  *Contributions to Mineralogy and Petrology*, 48, 179–203.
  https://doi.org/10.1007/BF00392328
- Thomas, J. B., & Rana, S. (2024). Garnet–clinopyroxene thermometry (as discussed alongside
  Yavuz & Yıldırım). Consult the source report for the exact calibration context; this notebook
  does not implement an unverified Powell equation.
  (The Ganguly piecewise coefficients above are attributed explicitly to Yavuz & Yıldırım.)